In [2]:
!wget https://raw.githubusercontent.com/Viishnu07/DLI_GroupAE_URLPhishing/main/Venus_Imrpoved%20CNN_Model/Venus_CNN.keras

--2025-08-24 13:37:20--  https://raw.githubusercontent.com/Viishnu07/DLI_GroupAE_URLPhishing/main/Venus_Imrpoved%20CNN_Model/Venus_CNN.keras
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 11510798 (11M) [application/octet-stream]
Saving to: ‘Venus_CNN.keras’

Venus_CNN.keras     100%[===================>]  10.98M  --.-KB/s    in 0.03s   

2025-08-24 13:37:20 (319 MB/s) - ‘Venus_CNN.keras’ saved [11510798/11510798]



In [8]:
import gradio as gr
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing.sequence import pad_sequences
import numpy as np
import string
import re
from urllib.parse import urlparse

# Load trained model
try:
    model = load_model("Venus_CNN.keras")
    print("Model loaded successfully!")
except Exception as e:
    print(f"Error loading model: {e}")
    model = None

# FIXED: Use the same character mapping as training
CHARS = string.printable
char2idx = {c: i + 1 for i, c in enumerate(CHARS)}  # 0 = padding
vocab_size = len(char2idx) + 1

MAX_LEN = 200

def preprocess_url(url):
    """Enhanced URL preprocessing"""
    # Remove leading/trailing whitespace
    url = url.strip()

    # Add protocol if missing
    if not url.startswith(('http://', 'https://')):
        url = 'http://' + url

    # Convert to lowercase for consistency
    url = url.lower()

    # Remove trailing slash if present
    url = url.rstrip('/')

    return url

def extract_url_features(url):
    """Extract additional features from URL that might help with classification"""
    try:
        parsed = urlparse(url)

        features = {
            'length': len(url),
            'domain_length': len(parsed.netloc) if parsed.netloc else 0,
            'path_length': len(parsed.path) if parsed.path else 0,
            'num_dots': url.count('.'),
            'num_hyphens': url.count('-'),
            'num_underscores': url.count('_'),
            'num_digits': sum(c.isdigit() for c in url),
            'has_ip': bool(re.match(r'\d+\.\d+\.\d+\.\d+', parsed.netloc)) if parsed.netloc else False,
            'suspicious_tld': parsed.netloc.endswith(('.tk', '.ml', '.ga', '.cf')) if parsed.netloc else False,
            'has_suspicious_keywords': any(word in url.lower() for word in
                                         ['secure', 'account', 'update', 'confirm', 'verify', 'login'])
        }
        return features
    except:
        return None

def encode_url(url, maxlen=MAX_LEN):
    """URL encoding to match training exactly"""
    # Use the exact same function as training
    sequence = [char2idx.get(c, 0) for c in url[:MAX_LEN]]
    return pad_sequences([sequence], maxlen=maxlen, padding='post', truncating='post')

def classify_text(user_input):
    """Classification with exact training match"""
    if not user_input.strip():
        return "Please enter a URL"

    if model is None:
        return "Model not loaded - check model file"

    try:
        # Use exact same encoding as training (no preprocessing)
        processed = encode_url(user_input)

        # Make prediction
        probs = model.predict(processed, verbose=0)[0]
        pred_class = np.argmax(probs)
        confidence = float(np.max(probs))

        # Determine label (assuming 0=legitimate, 1=phishing based on to_categorical usage)
        label = "Phishing" if pred_class == 1 else "Legitimate"

        # Create clean output
        result = f"**Prediction:** {label}\n"
        result += f"**Confidence:** {confidence:.3f}\n"
        result += f"**Probabilities:** [Legitimate: {probs[0]:.3f}, Phishing: {probs[1]:.3f}]\n\n"

        # Add some basic URL info
        features = extract_url_features(user_input)
        if features:
            result += f"**URL Length:** {features['length']} characters\n"
            result += f"**Domain:** {urlparse(user_input).netloc if '://' in user_input else 'Invalid format'}\n"

        return result

    except Exception as e:
        return f"Error during prediction: {str(e)}"

def debug_classify(user_input):
    """Debug version with detailed analysis"""
    if not user_input.strip():
        return "Please enter a URL"

    if model is None:
        return "Model not loaded - check model file"

    try:
        # Encode exactly as training
        processed = encode_url(user_input)

        # Make prediction
        probs = model.predict(processed, verbose=0)[0]
        pred_class = np.argmax(probs)
        confidence = float(np.max(probs))

        # Create detailed debug output
        result = f"**🔍 DEBUG ANALYSIS**\n\n"
        result += f"**URL:** {user_input}\n"
        result += f"**Character Set:** string.printable ({len(CHARS)} chars)\n"
        result += f"**Vocab Size:** {vocab_size}\n\n"

        # Show character mapping for first 20 chars
        result += f"**Character Encoding (first 20 chars):**\n"
        for i, char in enumerate(user_input[:20]):
            idx = char2idx.get(char, 0)
            result += f"  '{char}' -> {idx}\n"

        result += f"\n**Model Output:**\n"
        result += f"• Raw probabilities: {probs}\n"
        result += f"• Predicted class: {pred_class}\n"
        result += f"• Confidence: {confidence:.4f}\n\n"

        result += f"**Interpretations:**\n"
        result += f"• Class 0=Legit, 1=Phishing: **{'Phishing' if pred_class == 1 else 'Legitimate'}**\n"
        result += f"• Class 0=Phishing, 1=Legit: **{'Legitimate' if pred_class == 1 else 'Phishing'}**\n\n"

        result += f"**Sequence Info:**\n"
        result += f"• Input length: {len(user_input)}\n"
        result += f"• Padded shape: {processed.shape}\n"
        result += f"• First 15 encoded values: {processed[0][:15].tolist()}\n"

        return result

    except Exception as e:
        return f"Error: {str(e)}"
    """Get detailed model information"""
    if model is None:
        return "Model not loaded"

    try:
        info = []
        info.append("**MODEL INFORMATION**\n")
        info.append(f"Model type: {type(model).__name__}")
        info.append(f"Input shape: {model.input_shape}")
        info.append(f"Output shape: {model.output_shape}")
        info.append(f"Number of layers: {len(model.layers)}")
        info.append("")

        info.append("**LAYER DETAILS:**")
        for i, layer in enumerate(model.layers[:5]):  # First 5 layers
            info.append(f"  {i+1}. {layer.__class__.__name__}: {getattr(layer, 'output_shape', 'N/A')}")

        if len(model.layers) > 5:
            info.append(f"  ... and {len(model.layers) - 5} more layers")

        info.append("")
        info.append("**TRAINING INFO (if available):**")
        if hasattr(model, 'history'):
            info.append(f"Training history available: {bool(model.history)}")

        # Try to get model summary
        try:
            import io
            import sys
            old_stdout = sys.stdout
            sys.stdout = buffer = io.StringIO()
            model.summary()
            summary = buffer.getvalue()
            sys.stdout = old_stdout
            info.append("\n**MODEL SUMMARY:**")
            info.append(summary)
        except:
            info.append("Could not generate model summary")

        return "\n".join(info)
    except Exception as e:
        return f"Error getting model info: {e}"
    """Test different alphabet configurations"""
    test_alphabets = {
        "Current": string.ascii_letters + string.digits + ":/.?=-_&%+@#[](){},!~*;|",
        "Basic": string.ascii_letters + string.digits + ":/.?=-_",
        "Minimal": string.ascii_lowercase + string.digits + ":/.?=-_",
        "Extended": string.printable.replace(' \t\n\r\x0b\x0c', ''),  # All printable except whitespace
    }

    test_url = "https://www.google.com"

    results = []
    results.append("**ALPHABET CONFIGURATION TEST**\n")
    results.append(f"Testing URL: {test_url}\n")

    for name, alphabet_test in test_alphabets.items():
        char2idx_test = {ch: idx+1 for idx, ch in enumerate(alphabet_test)}

        # Test encoding
        seq = [char2idx_test.get(ch, 0) for ch in test_url.lower()]
        unknown_count = seq.count(0)

        results.append(f"**{name} Alphabet ({len(alphabet_test)} chars):**")
        results.append(f"  Characters: '{alphabet_test[:50]}{'...' if len(alphabet_test) > 50 else ''}'")
        results.append(f"  Unknown chars: {unknown_count}")
        results.append(f"  Sequence: {seq[:20]}{'...' if len(seq) > 20 else ''}")
        results.append("")

    return "\n".join(results)

# Enhanced Gradio interface
with gr.Blocks(title="AE Phishing Detector") as demo:
    gr.Markdown("# AE Phishing Detector - Enhanced Version\n*A Group Project with Improved Analysis*")

    with gr.Row():
        with gr.Column(scale=2):
            input_box = gr.Textbox(
                label="Enter URL",
                placeholder="Type or paste a URL (e.g., https://www.example.com)",
                lines=2
            )

            with gr.Row():
                predict_btn = gr.Button("Analyze URL", variant="primary")
                debug_btn = gr.Button("Debug Analysis")
                clear_btn = gr.Button("Clear")

        with gr.Column(scale=3):
            output_box = gr.Textbox(
                label="Analysis Results",
                lines=15,
                max_lines=20
            )

    with gr.Row():
        test_btn = gr.Button("Run Alphabet Test")
        model_info_btn = gr.Button("Show Model Info")

    # Event handlers
    predict_btn.click(classify_text, inputs=input_box, outputs=output_box)
    debug_btn.click(debug_classify, inputs=input_box, outputs=output_box)
    input_box.submit(classify_text, inputs=input_box, outputs=output_box)
    clear_btn.click(lambda: ("", ""), outputs=[input_box, output_box])

    # Example URLs
    gr.Markdown("""
    ## Example URLs to test:
    - **Legitimate:** `https://www.google.com`, `https://www.github.com`, `https://stackoverflow.com`
    - **Potentially Suspicious:** `http://secure-update.tk`, `https://verify-account-now.com/login`
    """)

if __name__ == "__main__":
    demo.launch(server_name="0.0.0.0", share=True)

Model loaded successfully!
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://8c3b1edc8ced9a951d.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
